# RoBERTa Fine-tuning — Creativity Score Prediction
---
This notebook fine-tunes **RoBERTa-base** to predict creativity scores from raw story text.

**Architecture:** `[Story Text] → RoBERTa Tokenizer → RoBERTa-base → [CLS] → Linear(768→1) → Score (0–10)`

⚙️ **Runtime:** Set to **GPU (T4)** via `Runtime → Change runtime type → T4 GPU`

⚠️ **Before running:** Upload `train.parquet`, `val.parquet`, and `test.parquet` to your Google Drive under a folder called `AICreativityJudge/data/`.

⏱️ **Expected time:** ~2-4 hours on a T4 GPU


In [ ]:
# ── Install & Imports ──
!pip install -q transformers datasets accelerate pandas pyarrow scikit-learn

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import time, os, gc

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/AICreativityJudge/data'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── Load Data ──
train_df = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/val.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/test.parquet')

TEXT_COL = 'story_truncated'
TARGET_COL = 'composite_score'

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"\nComposite score stats (train):")
print(train_df[TARGET_COL].describe().round(2))


In [ ]:
# ── PyTorch Dataset ──
class StoryDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_length=512):
        self.texts = texts
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.scores[idx], dtype=torch.float32)
        }

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

train_dataset = StoryDataset(
    train_df[TEXT_COL].tolist(),
    train_df[TARGET_COL].tolist(),
    tokenizer
)
val_dataset = StoryDataset(
    val_df[TEXT_COL].tolist(),
    val_df[TARGET_COL].tolist(),
    tokenizer
)
test_dataset = StoryDataset(
    test_df[TEXT_COL].tolist(),
    test_df[TARGET_COL].tolist(),
    tokenizer
)

BATCH_SIZE = 16  # T4 can handle 16 with RoBERTa-base
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=2)

print(f"Batches per epoch: {len(train_loader)}")
print("Datasets ready.")


In [ ]:
# ── Load RoBERTa for Regression ──
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=1,           # regression (single continuous output)
    problem_type='regression'
)
model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
# ── Training Config ──
EPOCHS = 3
LR = 2e-5
WARMUP_RATIO = 0.1

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")


In [ ]:
# ── Training Loop ──
train_losses = []
val_losses = []
best_val_loss = float('inf')
save_path = f'{DATA_DIR}/roberta_best.pt'

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    epoch_loss = 0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

        if (step + 1) % 200 == 0:
            elapsed = time.time() - t0
            steps_per_sec = (step + 1) / elapsed
            eta_min = (len(train_loader) - step - 1) / steps_per_sec / 60
            print(f"  Epoch {epoch+1} step {step+1}/{len(train_loader)}  "
                  f"loss={loss.item():.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}  "
                  f"ETA: {eta_min:.0f}min")

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validate ──
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    elapsed = time.time() - t0
    print(f"\nEpoch {epoch+1}/{EPOCHS}  "
          f"train_loss={avg_train_loss:.4f}  "
          f"val_loss={avg_val_loss:.4f}  "
          f"time={elapsed/60:.1f}min")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), save_path)
        print(f"  → Saved best model (val_loss={best_val_loss:.4f})")

    # Free up memory
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")


In [ ]:
# ── Training Curves ──
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS+1), train_losses, 'o-', label='Train Loss', color='#00d287')
plt.plot(range(1, EPOCHS+1), val_losses, 'o-', label='Val Loss', color='#00e5ff')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('RoBERTa Training Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(range(1, EPOCHS+1))
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/roberta_training_curves.png', dpi=150)
plt.show()


In [ ]:
# ── Evaluate on Test Set ──
model.load_state_dict(torch.load(save_path))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.squeeze(-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)

mse  = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print("=" * 50)
print("  RoBERTa — TEST SET RESULTS")
print("=" * 50)
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  R²:   {r2:.4f}")
print("=" * 50)

# Scatter plot
plt.figure(figsize=(8, 8))
plt.scatter(y_true, y_pred, alpha=0.05, s=5, color='#00e5ff')
plt.plot([0, 10], [0, 10], 'r--', alpha=0.7, label='Perfect')
plt.xlabel('True Composite Score')
plt.ylabel('Predicted Composite Score')
plt.title(f'RoBERTa: Predicted vs True (R²={r2:.3f})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/roberta_scatter.png', dpi=150)
plt.show()


In [ ]:
# ── Save Model for Inference ──
model.save_pretrained(f'{DATA_DIR}/roberta_creativity_model')
tokenizer.save_pretrained(f'{DATA_DIR}/roberta_creativity_model')

# Also save the test metrics
import json
metrics = {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}
with open(f'{DATA_DIR}/roberta_test_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Model saved to {DATA_DIR}/roberta_creativity_model/")
print(f"Metrics saved to {DATA_DIR}/roberta_test_metrics.json")
print("\n✅ Done! Download the model folder to your local project.")
